uv add sqlalchemy

# SQLAlchemy (ORM - Object Relational Mapping)
- Maps Python classes to database tables
- Typically, each database requires its own connector
- SQLAlchemy abstracts this by providing a unified interface across databases

# Important imports and explanations

from sqlalchemy import create_engine
- Responsible for creating the connection between your code and the database
  (e.g., MariaDB, PostgreSQL, SQLite, MySQL, Oracle)

from sqlalchemy.orm import declarative_base
- Used to define Python classes as database tables (ORM mapping)
- Base = declarative_base()
- Pass Base as the parent class when defining models

# To define a table in a Python class:
__tablename__ = "table_name"

from sqlalchemy import Column, String, Integer
# Example:
name = Column(String(50))

# Create tables in the database
Base.metadata.create_all(engine)
- Creates database objects based on defined Python models

from sqlalchemy.orm import sessionmaker
- Responsible for creating a database session (used to interact with the DB)

SessionBase = sessionmaker(engine)
session = SessionBase()

# Use the session to:
- Insert data
- Query data
- Commit transactions

# Always close the session after use
session.close()

# Alternatively, use context managers or try/finally to ensure proper closing

# from sqlalchemy.orm import RelationShip ('classname',backref, lazy='subquery')

In [97]:
#imports

from typing import List, Optional
from sqlalchemy import create_engine, Column, String, Integer, Boolean, delete, update
from sqlalchemy.orm import declarative_base, sessionmaker
from pathlib import Path

In [98]:
#remove database
database_path = Path("sqlalchemy.db")
if database_path.exists():
    database_path.unlink()


In [99]:
#engine / declarative_base
try:
    engine = create_engine(url="sqlite:///sqlalchemy.db")
    Base = declarative_base()
except Exception as e:
    print(e)
else:
    print("Engine and declarative base created successfully")

Engine and declarative base created successfully


In [100]:
#table users
class User(Base):
    __tablename__ = "users"
    
    id = Column(Integer,primary_key=True,autoincrement=True)
    name = Column(String(50), nullable=False)
    admin = Column(Boolean, default=False)

Base.metadata.create_all(engine)

In [101]:
#database session default
database_session_maker = sessionmaker(engine)
#session = database_session_maker() -- alternative

In [102]:
#add user / session maker using with, no necessary session.close()

list_users: List[User] = []

user_leonardo = User(name = "Leonardo",admin=True)
user_john = User(name="John", admin=False)
user_mike = User(name="Mike", admin=True)

list_users.append(user_leonardo)
list_users.append(user_john)
list_users.append(user_mike)

with database_session_maker() as session:
    for user in list_users:
        session.add(user)
        session.commit()

In [103]:
#select query
with database_session_maker() as session:
    users = session.query(User).all()

for user in users:
    print(f"{user.id} -> {user.name} -> {user.admin}")

1 -> Leonardo -> True
2 -> John -> False
3 -> Mike -> True


In [104]:
#update

with database_session_maker() as session:
    update_user = session.query(User).filter_by(id = 1).first()
    update_user.name = "Leo - Updated"
    session.commit()


In [105]:
#using order by

with database_session_maker() as session:
    results_query = session.query(User).order_by(User.id.asc()).all()
    for results in results_query:
        print(f"{results.id} -> {results.name} -> {results.admin}")

1 -> Leo - Updated -> True
2 -> John -> False
3 -> Mike -> True


In [ ]:
#add table: professions - relationship

class Table(Base):
    __tablename__ = "professions"
    
    id = Column(Integer, primary_key=True, nullable=False, autoincrement=True)
    name = Column(String, unique=True, nullable=False)
    
#Base.metadata.create_all(engine)